In [1]:
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
import pandas as pd


ds = fetch_ucirepo(id=320)

In [2]:
from sklearn.preprocessing import KBinsDiscretizer


# Our stated target is grade category so we preprocess the target accordingly
pp = KBinsDiscretizer(encode="ordinal")
pp.set_output(transform="pandas")

y = pp.fit_transform(ds["data"]["targets"]["G3"].to_frame()).astype(int)
y.value_counts()

G3
2     176
3     145
4     131
0     100
1      97
Name: count, dtype: int64

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    ds["data"]["features"][[
        "famsup",
        "higher",
        "school",
        "failures"
    ]],
    y,
    test_size=0.2,
    random_state=42
)
pd.concat([X_train, y_train], axis=1)

,famsup,higher,school,failures,G3
332,yes,yes,GP,0,4
29,yes,yes,GP,0,2
302,no,yes,GP,0,2
286,no,yes,GP,0,3
554,yes,no,MS,0,1
...,...,...,...,...,...
71,no,yes,GP,0,1
106,yes,yes,GP,0,1
270,no,yes,GP,0,4
435,yes,yes,MS,0,1


In [4]:
cpt = (
    pd.concat([X_train, y_train], axis=1)
    .groupby(list(X_train.columns))[list(y_train.columns)]
    .value_counts(normalize=True)
)
cpt.to_csv("cpt.csv")

pred_table = (
    pd.concat([X_train, y_train], axis=1)
    .groupby(list(X_train.columns))[y_train.columns[0]]
    .agg(lambda x: x.mode()[0])
)
pred_table

famsup  higher  school  failures
no      no      GP      0           0
                        1           0
                        2           0
                        3           0
                MS      0           0
                        1           0
                        3           0
        yes     GP      0           3
                        1           1
                        2           1
                        3           1
                MS      0           2
                        1           0
                        2           0
                        3           0
yes     no      GP      0           2
                        1           0
                        2           1
                        3           0
                MS      0           0
                        1           0
                        2           1
                        3           0
        yes     GP      0           3
                        1           1
                 

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error


y_test_pred = []
for _, X in X_test.iterrows():
    feats = X[list(X_train.columns)]
    y_test_pred.append(pred_table.get(tuple(feats), 1))

print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_test_pred))

# It's generally not a good idea to calculate MAE on a categorical target,
# but it helps me gauge if the model is at least close
print("MAE:", mean_absolute_error(y_test, y_test_pred))

Accuracy: 0.3
Confusion matrix:
 [[ 7  2  4  2  0]
 [ 6  1  5  5  0]
 [ 2  4 10 25  0]
 [ 0  0  4 21  0]
 [ 0  1 11 20  0]]
MAE: 0.9153846153846154
